In [ ]:
wip_source AS (
    SELECT DISTINCT
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN LOWER(COALESCE(st.description, '')) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
            WHEN e.activity_date_time > current_timestamp() THEN 'Booked'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
            ELSE 'Unknown'
        END AS session_status_src_name,

        CONCAT('WIP001_', LOWER(TRIM(CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN LOWER(COALESCE(st.description, '')) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
            WHEN e.activity_date_time > current_timestamp() THEN 'Booked'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
            ELSE 'Unknown'
        END))) AS session_status_src_id,

        'WIP001' AS session_status_src_sys_inst_id

    FROM silver_wip_activityentry e
    LEFT JOIN silver_wip_activityheader h
        ON e.activity_header_id = h.id
    LEFT JOIN silver_wip_activitystatus st
        ON h.activity_status_id = st.id

    WHERE e.activity_date_time IS NOT NULL
       OR e.is_dna = true
       OR LOWER(COALESCE(st.description, '')) LIKE '%cancel%'
),